In [ ]:
# @title 1 · Environment + /kaggle/tmp storage
# ═══════════════════════════════════════════════════════════════
#  WeeLLM Studio · Kaggle · Cell 1 of 2
#  Settings → Accelerator → GPU (T4 x2 / P100), then Run All.
#  All model downloads are routed to /kaggle/tmp and the default HF
#  cache is symlinked there, so the big weights never fill /kaggle/working.
#  Gated repos (Ideogram, LTX) need an HF token: add it as a Kaggle Secret
#  named HF_TOKEN (Add-ons → Secrets) and Agree/Accept on the repo page.
# ═══════════════════════════════════════════════════════════════
import os, sys, subprocess, time, shutil, warnings, contextlib, io
warnings.simplefilter('ignore')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# ── route downloads to /kaggle/tmp and link the HF cache there ──
TMP = '/kaggle/tmp'
HF_HOME = os.path.join(TMP, 'huggingface')
os.makedirs(os.path.join(HF_HOME, 'hub'), exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
os.environ['HF_HUB_CACHE'] = os.path.join(HF_HOME, 'hub')
_default_cache = os.path.expanduser('~/.cache/huggingface')
os.makedirs(os.path.dirname(_default_cache), exist_ok=True)
if not os.path.exists(_default_cache):
    os.symlink(HF_HOME, _default_cache)   # default-path Hub calls also land on tmp
OUT_DIR = '/kaggle/working/weellm_studio'
os.makedirs(OUT_DIR, exist_ok=True)
os.environ['WEELLM_OUT_DIR'] = OUT_DIR
print('storage: HF cache -> %s (linked from %s) · outputs -> %s' % (os.environ['HF_HUB_CACHE'], _default_cache, OUT_DIR))

import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Set Settings → Accelerator → GPU, then re-run.')
print('gpu:', torch.cuda.get_device_name(0))

# torchao must expose quantization.FqnToConfig; match the wheel to torch's CUDA.
_parts = (torch.version.cuda or 'cpu').split('.')
_tag = 'cu' + _parts[0] + _parts[1] if _parts[0].isdigit() else 'cpu'
_IDX = 'https://download.pytorch.org/whl/' + _tag
STEPS = [
    ('Base tools (hf_transfer · gradio · peft)', [sys.executable, '-m', 'pip', 'install', '-q', 'hf_transfer', 'psutil', 'gradio>=5', 'peft>=0.10.0']),
    ('torchao upgrade', [sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torchao', '--index-url', _IDX]),
    ('diffusers @ git', [sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/huggingface/diffusers.git']),
    ('WeeLLM-Enhanced @ git (fresh, no deps)', [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps', 'git+https://github.com/GodL-x-SouL/WeeLLM-Enhanced.git']),
    ('WeeLLM deps (fill gaps)', [sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'safetensors', 'numpy', 'Pillow', 'tqdm', 'gguf', 'av', 'huggingface_hub']),
]
t0 = time.time()
ok_all = True
for name, cmd in STEPS:
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=1500)
    ok = (p.returncode == 0)
    ok_all = ok_all and ok
    print('%-42s %s' % (name, 'done' if ok else 'FAILED: ' + (p.stderr or '')[-300:]), flush=True)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        try:
            import gradio, diffusers, huggingface_hub, weellm
            from torchao import quantization as _tq
            assert hasattr(_tq, 'FqnToConfig'), 'torchao too old: pip install -U torchao'
            import av  # video export
            free = shutil.disk_usage(TMP).free / 1e9
            print('verify: torch %s · gradio %s · diffusers %s · weellm %s · /kaggle/tmp free %.0f GB · %.0fs'
                  % (torch.__version__, gradio.__version__, diffusers.__version__, weellm.__version__, free, time.time() - t0))
        except Exception as e:
            ok_all = False
            print('VERIFY FAILED: %s: %s' % (type(e).__name__, e))
print('CELL 1 %s — run Cell 2 to open the Studio.' % ('READY' if ok_all else 'FAILED (read rows above)'))


In [ ]:
# @title 2 · Studio server (keep this cell running)
# ═══════════════════════════════════════════════════════════════
#  WeeLLM Studio · Kaggle · Cell 2 of 2
#  The UI lives in the separate GradioUI.py — it is NOT embedded here.
#  This cell locates that file (working dir, attached Kaggle input, or the
#  GitHub repo), imports it, and serves it as a public Gradio share URL.
#  The heartbeat below holds the session open; stop the cell to shut down.
# ═══════════════════════════════════════════════════════════════
import os, sys, time, glob, shutil, urllib.request
WORK = '/kaggle/working'
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

# locate the separate GradioUI.py (never paste its source into this cell)
_target = os.path.join(WORK, 'GradioUI.py')
_cands = [_target] + sorted(glob.glob('/kaggle/input/**/GradioUI.py', recursive=True))
_src = next((c for c in _cands if os.path.isfile(c)), None)
if _src is None:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/GodL-x-SouL/WeeLLM-Enhanced/main/GradioUI.py', _target)
elif os.path.abspath(_src) != os.path.abspath(_target):
    shutil.copyfile(_src, _target)
print('ui: %s' % _target)

import GradioUI as studio
_url = getattr(studio, '_URL', None)
if not _url:
    _url = studio.serve()          # share=True on Kaggle; returns the public URL
    studio._URL = _url
from IPython.display import display, HTML
display(HTML(
    "<div style='max-width:640px;margin:12px auto;padding:16px;border-radius:10px;"
    "background:#15151a;border:1px solid #26262e;color:#e8e8ea;font-family:system-ui,sans-serif'>"
    "<div style='font-size:14px;font-weight:600;margin-bottom:8px'>WeeLLM Studio · live</div>"
    "<a href='%s' target='_blank' style='display:inline-block;padding:9px 22px;border-radius:6px;"
    "background:#e8e8ea;color:#101014;text-decoration:none;font-size:13px;font-weight:600'>OPEN STUDIO</a>"
    "<div style='margin-top:8px;font-size:11px;color:#8a8a95;word-break:break-all'>%s</div>"
    "<div style='margin-top:6px;font-size:11px;color:#8a8a95'>Keep this cell running · models stream from /kaggle/tmp</div></div>"
    % (_url, _url)))
print(_url, flush=True)

import torch
_t0, _beats = time.time(), 0
while True:  # keep-alive: hold the cell (and the Kaggle session) awake while serving
    time.sleep(60)
    _beats += 1
    _up = int(time.time() - _t0)
    try:
        _vram = torch.cuda.memory_allocated() / 1e9
    except Exception:
        _vram = -1.0
    print('heartbeat #%d · uptime %dh %02dm · VRAM %.2f GB · serving — stop the cell to shut down'
          % (_beats, _up // 3600, (_up % 3600) // 60, _vram), flush=True)
